# 01 · Las cinco viviendas más alejadas de la mediana por costa · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Retomá el livecoding anterior: para cada `ocean_proximity`, calculá la mediana de
`median_house_value` y extraé las **cinco filas cuya distancia absoluta a esa mediana sea mayor**.
Recordá que cada fila representa una zona censal.

- Usá todas las filas de la base preparada, incluidos los precios en el máximo observado.
- Devolvé las columnas originales, seguidas por `mediana_grupo` y `distancia_mediana`.
- Conservá el índice original. Ordená las categorías alfabéticamente y, dentro de cada una,
  la distancia de mayor a menor. Si hay empate, conservá el orden original de las filas.
- `n` es un entero positivo. Si un grupo tiene menos filas, devolvé todas las disponibles.
- Una entrada vacía debe devolver un DataFrame vacío con las columnas indicadas.
- No modifiques el DataFrame recibido.

**Ejemplo:** precios `[100, 200, 300, 400, 1000]` → mediana `300` → distancias
`[200, 100, 0, 100, 700]`. Con `n=2`, se extraen las filas de `1000` y `100`.

## 3. Resolución comentada

1. `copy()` permite agregar columnas sin alterar la entrada.
2. `transform('median')` devuelve una mediana por fila, correspondiente a su costa.
3. `abs()` mide la distancia tanto por encima como por debajo de esa mediana.
4. Ordenamos por distancia y usamos `head(n)` dentro de cada grupo. El orden estable
   conserva el desempate por posición original, incluso después de ordenar las costas.

El resultado conserva filas completas e índices. Para consultar las 25 filas reales,
podés ejecutar `display(resultado)`; la vista guardada muestra las primeras 20.

In [3]:
def extraer_extremos_por_costa(datos, n=5):
    """Extrae las n filas más alejadas de la mediana de cada categoría de costa."""
    resultado = datos.copy()

    # transform repite la mediana del grupo en cada una de sus filas.
    resultado["mediana_grupo"] = resultado.groupby("ocean_proximity")[
        "median_house_value"
    ].transform("median")
    resultado["distancia_mediana"] = (
        resultado["median_house_value"] - resultado["mediana_grupo"]
    ).abs()

    # El orden estable conserva la posición original cuando hay empates.
    ordenado = resultado.sort_values(
        "distancia_mediana", ascending=False, kind="stable"
    )
    extremos = ordenado.groupby("ocean_proximity", sort=False).head(n)

    # Agrupamos visualmente las categorías sin alterar el orden dentro de ellas.
    return extremos.sort_values("ocean_proximity", kind="stable")

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "ocean_proximity": ["B", "A", "A", "B", "A", "C", "B"],
        "median_house_value": [10, 100, 0, 20, 50, 7, 30],
        "identificador": list("abcdefg"),
    }, index=[90, 7, 42, 3, 8, 55, 1])
    original = caso.copy(deep=True)

    # Empates, grupos con menos de n filas y conservación de las columnas originales.
    esperado = caso.iloc[[1, 2, 0, 6, 5]].copy()
    esperado["mediana_grupo"] = [50, 50, 20, 20, 7]
    esperado["distancia_mediana"] = [50, 50, 10, 10, 0]
    comparar(funcion(caso, n=2), esperado)
    comparar(caso, original)

    comparar(funcion(caso, n=1), esperado.iloc[[0, 2, 4]])

    esperado_todos = caso.iloc[[1, 2, 4, 0, 6, 3, 5]].copy()
    esperado_todos["mediana_grupo"] = [50, 50, 50, 20, 20, 20, 7]
    esperado_todos["distancia_mediana"] = [50, 50, 0, 10, 10, 0, 0]
    comparar(funcion(caso, n=99), esperado_todos)

    vacio = caso.iloc[:0].copy()
    esperado_vacio = vacio.copy()
    esperado_vacio["mediana_grupo"] = pd.Series(dtype=float)
    esperado_vacio["distancia_mediana"] = pd.Series(dtype=float)
    comparar(funcion(vacio, n=5), esperado_vacio)
    comparar(caso, original)
    print("OK: extremos, empates, n variable, grupo pequeño, entrada vacía y datos intactos.")


probar(extraer_extremos_por_costa)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = extraer_extremos_por_costa(datos)
comparar(datos, antes)
assert list(resultado.columns) == list(datos.columns) + ["mediana_grupo", "distancia_mediana"]
assert len(resultado) == datos.groupby("ocean_proximity").size().clip(upper=5).sum()
for grupo, filas in resultado.groupby("ocean_proximity"):
    valores = datos.loc[datos["ocean_proximity"] == grupo, "median_house_value"]
    distancias = np.sort(np.abs(valores.to_numpy() - np.median(valores)))[::-1][:5]
    np.testing.assert_allclose(filas["distancia_mediana"], distancias)
    np.testing.assert_allclose(filas["mediana_grupo"], np.median(valores))
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: extremos, empates, n variable, grupo pequeño, entrada vacía y datos intactos.
OK: dataset real. Resultado: 25 filas.


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household,mediana_grupo,distancia_mediana
923,-121.89,37.49,9.0,4909.0,577.0,1981.0,591.0,9.7194,500001.0,<1H OCEAN,False,8.306261,0.117539,3.351946,214850.0,285151.0
955,-121.93,37.66,24.0,3166.0,424.0,1081.0,400.0,8.3337,500001.0,<1H OCEAN,False,7.915000,0.133923,2.702500,214850.0,285151.0
1582,-122.00,37.86,18.0,8953.0,1074.0,3011.0,993.0,10.7372,500001.0,<1H OCEAN,False,9.016113,0.119960,3.032226,214850.0,285151.0
1585,-122.00,37.84,16.0,7681.0,946.0,2777.0,908.0,9.5271,500001.0,<1H OCEAN,False,8.459251,0.123161,3.058370,214850.0,285151.0
3486,-118.54,34.28,10.0,7665.0,999.0,3517.0,998.0,10.8805,500001.0,<1H OCEAN,False,7.680361,0.130333,3.524048,214850.0,285151.0
1583,-121.97,37.87,4.0,1029.0,126.0,416.0,122.0,13.4883,500001.0,INLAND,False,8.434426,0.122449,3.409836,108500.0,391501.0
1586,-121.96,37.85,10.0,3209.0,379.0,1199.0,392.0,12.2478,500001.0,INLAND,False,8.186224,0.118105,3.058673,108500.0,391501.0
1644,-121.94,37.83,11.0,2836.0,373.0,959.0,335.0,10.5815,500001.0,INLAND,False,8.465672,0.131523,2.862687,108500.0,391501.0
1645,-121.89,37.82,4.0,11444.0,1355.0,3898.0,1257.0,13.2949,500001.0,INLAND,False,9.104216,0.118403,3.101034,108500.0,391501.0
1646,-121.91,37.81,7.0,3477.0,416.0,1216.0,395.0,13.1499,500001.0,INLAND,False,8.802532,0.119643,3.078481,108500.0,391501.0


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.